In [ ]:
"""
Brain Aging Analysis Pipeline (Clean Version)
==============================================
1. Data loading & preprocessing
2. GLM regression (parallel)
3. LOWESS fitting (parallel, for visualization only)
4. FDR correction + beta-direction consistency filtering
5. Visualization
   - Single-outcome heatmap
   - Consistent-feature heatmap
   - UpSet intersection plot
   - 4-column aligned heatmap
"""

import os
import time
import gc
from itertools import combinations
from multiprocessing import Pool, cpu_count

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr
from scipy.interpolate import PchipInterpolator
from scipy.ndimage import gaussian_filter1d


# ========================= Configuration =========================
DATA_PATH = "./data/work/0309fourbrain_behavior_cov.csv"
MATCH_PATH = "./data/work/column_match_result.csv"
SAVE_DIR = "./results/BrainAging_Clean"
FDR_ALPHA = 0.05
N_JOBS = 80

OUTCOME_COLS = [
    "brain_difference",
    "baizhi_difference",
    "huizhi_difference",
    "protein_difference",
]
OUTCOME_LABELS = {
    "brain_difference": "Brain",
    "baizhi_difference": "White Matter",
    "huizhi_difference": "Gray Matter",
    "protein_difference": "Protein",
}

GLM_COVARIATES = ["age", "sex", "race", "APOE4_carrier"]

MANUAL_CATEGORY = {
    "Glycated.haemoglobin..HbA1c": "Biochemical markers",
    "summed_MET": "Lifestyles",
    "Average_total_household_income_before_tax": "Psychosocial factors",
}
CATEGORY_ORDER = [
    "Biochemical markers",
    "Body measure",
    "Lifestyles",
    "Food Intake",
    "Psychosocial factors",
]

# NPG color palette
CATEGORY_COLORS = {
    "Biochemical markers": "#E64B35",
    "Body measure":        "#4DBBD5",
    "Lifestyles":          "#00A087",
    "Food Intake":         "#F39B7F",
    "Psychosocial factors":"#3C5488",
}
UPSET_COLORS = {
    "all4":   "#E64B35",
    "three":  "#3C5488",
    "two":    "#4DBBD5",
    "single": "#B09C85",
}
COLOR_CONSISTENT = "#E64B35"
COLOR_DEFAULT    = "#3d3d3a"


# ========================= Utility Functions =========================

def make_diverging_cmap():
    """Create an RdBu diverging colormap with a white center."""
    colors = [
        mcolors.to_rgb("#2166AC"),
        mcolors.to_rgb("#F7F7F7"),
        mcolors.to_rgb("#B2182B"),
    ]
    return mcolors.LinearSegmentedColormap.from_list("RdBu_custom", colors, N=256)


def build_feature_category_map(match_path, manual_overrides):
    """Build a feature -> category mapping from the match file."""
    match_df = pd.read_csv(match_path)
    matched = match_df[match_df["CSV列名"].notna() & (match_df["CSV列名"] != "")]
    mapping = {}
    for _, row in matched.iterrows():
        csv_col = row["CSV列名"]
        cat = row["Category"]
        if pd.notna(cat) and cat != "":
            mapping[csv_col] = cat
    mapping.update(manual_overrides)
    return mapping


def select_features(df, feature_to_category, exclude_cols, min_obs=100):
    """Select valid feature columns with enough observations."""
    features = []
    for f in feature_to_category:
        if f in df.columns and f not in exclude_cols:
            df[f] = pd.to_numeric(df[f], errors="coerce")
            if df[f].notna().sum() >= min_obs:
                features.append(f)
    return features


# ========================= 1. GLM Regression =========================

def _glm_worker(args):
    """Worker function for a single GLM regression task."""
    outcome, feature, x, y, covs = args
    mask = ~np.isnan(x) & ~np.isnan(y)
    if covs is not None:
        mask &= ~np.any(np.isnan(covs), axis=1)
    n = mask.sum()
    min_n = (covs.shape[1] + 5) if covs is not None else 10
    if n < min_n:
        return outcome, feature, np.nan, np.nan
    x_m, y_m = x[mask], y[mask]
    sd = y_m.std()
    if sd == 0:
        return outcome, feature, 0.0, np.nan
    y_z = (y_m - y_m.mean()) / sd
    try:
        if covs is not None:
            X = np.column_stack([np.ones(n), x_m, covs[mask]])
        else:
            X = np.column_stack([np.ones(n), x_m])
        model = sm.OLS(y_z, X).fit()
        return outcome, feature, model.params[1], model.pvalues[1]
    except Exception:
        return outcome, feature, np.nan, np.nan


def run_glm(df, outcome_cols, features, glm_covariates, n_jobs):
    """Run GLM regression for all outcome x feature combinations in parallel."""
    print(f"\n[GLM] Starting ({n_jobs} cores)")
    t0 = time.time()

    cov_cols = [c for c in glm_covariates if c in df.columns]
    cov_matrix = df[cov_cols].values.astype(np.float64) if cov_cols else None

    outcome_arr = {
        oc: pd.to_numeric(df[oc], errors="coerce").values.astype(np.float64)
        for oc in outcome_cols
    }
    feature_arr = {
        f: pd.to_numeric(df[f], errors="coerce").values.astype(np.float64)
        for f in features
    }

    tasks = []
    for oc in outcome_cols:
        for f in features:
            if f == oc or f in cov_cols or f == "eid":
                continue
            tasks.append((oc, f, outcome_arr[oc], feature_arr[f], cov_matrix))

    print(f"  Total tasks: {len(tasks)}")
    chunksize = max(1, len(tasks) // (n_jobs * 4))

    try:
        with Pool(n_jobs) as pool:
            results = pool.map(_glm_worker, tasks, chunksize=chunksize)
    except Exception:
        results = [_glm_worker(t) for t in tasks]
    gc.collect()

    glm_results = {oc: {} for oc in outcome_cols}
    for oc, f, beta, pval in results:
        glm_results[oc][f] = (beta, pval)

    print(f"  Elapsed: {time.time() - t0:.1f}s")
    return glm_results


# ========================= 2. LOWESS Fitting =========================

def _loess_worker(args):
    """Worker function for a single LOWESS fitting task."""
    df_sub, outcome_col, feature, x_grid, frac, cov_cols = args
    if feature not in df_sub.columns:
        return feature, np.full(len(x_grid), np.nan)

    cols = [outcome_col, feature] + [c for c in cov_cols if c in df_sub.columns]
    sub = df_sub[cols].dropna()
    if sub.shape[0] < 20:
        return feature, np.full(len(x_grid), np.nan)

    x_s = sub[outcome_col].values
    y_s = sub[feature].values
    sd = y_s.std()
    if sd == 0:
        return feature, np.full(len(x_grid), 0.0)
    y_z = (y_s - y_s.mean()) / sd

    try:
        rho, _ = spearmanr(x_s, y_z)
        use_frac = 0.6 if abs(rho) < 0.05 else frac

        loess_out = lowess(y_z, x_s, frac=use_frac, return_sorted=True, it=0)
        pred = np.interp(x_grid, loess_out[:, 0], loess_out[:, 1])

        # Light smoothing via PCHIP interpolation + Gaussian filter
        valid = np.where(~np.isnan(pred))[0]
        if len(valid) >= 5:
            try:
                interp = PchipInterpolator(x_grid[valid], pred[valid])
                pred = gaussian_filter1d(interp(x_grid), sigma=2)
            except Exception:
                pass
        return feature, pred
    except Exception:
        return feature, np.full(len(x_grid), np.nan)


def run_lowess(df, outcome_cols, features, glm_covariates, n_grid=300,
               frac=0.5, n_jobs=80):
    """Run LOWESS fitting for all outcome x feature combinations."""
    print(f"\n[LOWESS] Starting ({n_jobs} cores)")
    all_loess = {}
    x_grids = {}

    cov_cols = [c for c in glm_covariates if c in df.columns]

    for oc in outcome_cols:
        print(f"  Fitting outcome: {oc}")
        use_cols = list(set([oc] + features + cov_cols))
        df_sub = df[use_cols].copy()
        df_sub[oc] = pd.to_numeric(df_sub[oc], errors="coerce")
        df_sub = df_sub.dropna(subset=[oc])
        for f in features:
            df_sub[f] = pd.to_numeric(df_sub[f], errors="coerce")

        x_grid = np.linspace(df_sub[oc].min(), df_sub[oc].max(), n_grid)
        x_grids[oc] = x_grid

        tasks = [(df_sub, oc, f, x_grid, frac, cov_cols) for f in features]
        nj = min(n_jobs, len(features), cpu_count())

        try:
            with Pool(nj, maxtasksperchild=1) as pool:
                results = pool.map(_loess_worker, tasks)
        except Exception:
            results = [_loess_worker(t) for t in tasks]
        gc.collect()

        all_loess[oc] = {f: pred for f, pred in results}
        print(f"    Done: {len(all_loess[oc])} features")

    return all_loess, x_grids


# ========================= 3. FDR Correction + Consistency =========================

def run_fdr(glm_results, outcome_cols, features, fdr_alpha=0.05):
    """FDR correction and beta-direction consistency filtering."""
    print(f"\n[FDR] alpha={fdr_alpha}")
    records = []
    for f in features:
        row = {"feature": f}
        for oc in outcome_cols:
            beta, p = glm_results[oc].get(f, (np.nan, np.nan))
            row[f"beta_{oc}"] = beta
            row[f"p_{oc}"] = p
        records.append(row)
    fdr_df = pd.DataFrame(records)

    for oc in outcome_cols:
        pvals = np.where(np.isnan(fdr_df[f"p_{oc}"].values), 1.0,
                         fdr_df[f"p_{oc}"].values)
        reject, fdr_pvals, _, _ = multipletests(pvals, alpha=fdr_alpha, method="fdr_bh")
        fdr_df[f"fdr_{oc}"] = fdr_pvals
        fdr_df[f"sig_{oc}"] = reject
        print(f"  {oc}: {reject.sum()} significant")

    sig_cols = [f"sig_{oc}" for oc in outcome_cols]
    beta_cols = [f"beta_{oc}" for oc in outcome_cols]

    fdr_df["all_sig"] = fdr_df[sig_cols].all(axis=1)

    def direction_consistent(row):
        signs = [np.sign(row[bc]) for bc in beta_cols if row[bc] != 0]
        return len(signs) == len(outcome_cols) and len(set(signs)) == 1

    fdr_df["dir_consistent"] = fdr_df.apply(direction_consistent, axis=1)
    fdr_df["consistent"] = fdr_df["all_sig"] & fdr_df["dir_consistent"]

    consistent_features = fdr_df[fdr_df["consistent"]]["feature"].tolist()
    print(f"  Significant in all 4 outcomes: {fdr_df['all_sig'].sum()}")
    print(f"  With consistent direction: {len(consistent_features)}")

    return consistent_features, fdr_df


# ========================= 4. Visualization =========================

def _compute_trend(pred_map):
    """Compute the overall trend (last - first) for each feature."""
    return {
        f: (pred[-1] - pred[0] if pred is not None and not np.all(np.isnan(pred)) else 0.0)
        for f, pred in pred_map.items()
    }


def _sort_by_category(features, feature_to_category, category_order, sort_key):
    """Sort features by category groups and within-group sort key.
    Returns (sorted_features, category_boundaries).
    """
    sorted_feats = []
    boundaries = []
    for cat in category_order:
        group = [f for f in features if feature_to_category.get(f, "") == cat]
        if not group:
            continue
        group.sort(key=sort_key, reverse=True)
        start = len(sorted_feats)
        sorted_feats.extend(group)
        boundaries.append((start, len(sorted_feats), cat))

    other = [f for f in features if feature_to_category.get(f, "") not in category_order]
    if other:
        other.sort(key=sort_key, reverse=True)
        start = len(sorted_feats)
        sorted_feats.extend(other)
        boundaries.append((start, len(sorted_feats), "Other"))

    return sorted_feats, boundaries


def _draw_category_strip(ax, boundaries, n_features, category_colors):
    """Draw a colored category strip on the left axis."""
    ax.set_xlim(0, 1)
    ax.set_ylim(n_features, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    for start, end, cat in boundaries:
        color = category_colors.get(cat, "#B09C85")
        rect = mpatches.FancyBboxPatch(
            (0, start), 1, end - start,
            boxstyle="round,pad=0",
            facecolor=color, edgecolor="white", linewidth=1.5,
        )
        ax.add_patch(rect)
        mid = (start + end) / 2
        ax.text(0.5, mid, cat, fontsize=7, fontweight="bold",
                color="white", ha="center", va="center", rotation=90)


def _build_heatmap_matrix(sorted_features, pred_map):
    """Assemble a heatmap matrix from pred_map. Returns (heat, n_grid)."""
    rows = []
    n_grid = None
    for f in sorted_features:
        p = pred_map.get(f)
        if p is not None and not np.all(np.isnan(p)):
            if n_grid is None:
                n_grid = len(p)
            rows.append(p.copy())
        else:
            rows.append(np.full(n_grid or 300, np.nan))
    heat = np.nan_to_num(np.array(rows), nan=0.0)
    return heat, n_grid


def _normalize_heat(heat):
    """Per-panel normalize heatmap values to [-1, 1]."""
    finite = heat[~np.isnan(heat)]
    if finite.size > 0:
        amax = max(abs(np.percentile(finite, 2)), abs(np.percentile(finite, 98)), 1e-6)
    else:
        amax = 1.0
    return heat / amax


# ---------- Plot 1: Single-outcome heatmap ----------

def plot_single_heatmap(outcome_col, features, pred_map, feature_to_category,
                        consistent_set, save_path=None):
    """Full-feature heatmap for a single outcome, with consistent features annotated."""
    valid = [f for f in features
             if not np.all(np.isnan(pred_map.get(f, np.array([np.nan]))))]
    if not valid:
        return

    trend_map = _compute_trend(pred_map)
    sorted_feats, boundaries = _sort_by_category(
        valid, feature_to_category, CATEGORY_ORDER,
        sort_key=lambda f: trend_map.get(f, 0),
    )

    heat, _ = _build_heatmap_matrix(sorted_feats, pred_map)
    heat_norm = _normalize_heat(heat)
    cmap = make_diverging_cmap()
    n_features = len(sorted_feats)

    has_marks = bool(consistent_set)
    fig_height = max(6, n_features * 0.06)

    if has_marks:
        fig, (ax_cat, ax_heat, ax_mark) = plt.subplots(
            1, 3, figsize=(22, fig_height),
            gridspec_kw={"width_ratios": [0.8, 16, 8], "wspace": 0.02},
        )
    else:
        fig, (ax_cat, ax_heat) = plt.subplots(
            1, 2, figsize=(14, fig_height),
            gridspec_kw={"width_ratios": [0.8, 20], "wspace": 0.02},
        )

    sns.heatmap(heat_norm, cmap=cmap, vmin=-1, vmax=1, center=0,
                yticklabels=False, xticklabels=False, cbar=False, ax=ax_heat)
    for start, end, cat in boundaries:
        if start > 0:
            ax_heat.axhline(y=start, color="white", linewidth=2.5)
    ax_heat.set_xlabel(f"{outcome_col} (younger -> older)", fontsize=12)
    ax_heat.set_title(f"{outcome_col}: Trend Heatmap by Category",
                      fontsize=14, fontweight="bold", pad=12)

    _draw_category_strip(ax_cat, boundaries, n_features, CATEGORY_COLORS)

    if has_marks:
        marked = [(i, f) for i, f in enumerate(sorted_feats) if f in consistent_set]
        n_marked = len(marked)
        ax_mark.set_xlim(0, 10)
        ax_mark.set_ylim(n_features, 0)
        ax_mark.set_xticks([])
        ax_mark.set_yticks([])
        for sp in ax_mark.spines.values():
            sp.set_visible(False)
        if n_marked > 0:
            fs = max(4, min(7, 500 / n_marked))
            gap = max(1.5, n_features / (n_marked * 1.5))
            label_ys = []
            for k, (row_idx, _) in enumerate(marked):
                y = row_idx + 0.5
                if label_ys:
                    y = max(y, label_ys[-1] + gap)
                label_ys.append(min(y, n_features - 0.5))
            for k, (row_idx, fname) in enumerate(marked):
                ax_mark.plot([0, 1], [row_idx + 0.5, label_ys[k]],
                             color="#8491B4", linewidth=0.5, alpha=0.6)
                ax_mark.text(1.2, label_ys[k], f"★ {fname}", fontsize=fs,
                             color=COLOR_CONSISTENT, fontweight="bold",
                             va="center", ha="left")

    # Colorbar
    sm_cb = ScalarMappable(cmap=cmap, norm=Normalize(vmin=-1, vmax=1))
    sm_cb.set_array([])
    cbar_ax = fig.add_axes([0.12, 0.02, 0.35, 0.012])
    fig.colorbar(sm_cb, cax=cbar_ax, orientation="horizontal").set_label(
        "Normalized Z-score", fontsize=9)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight", format="pdf")
    plt.close()


# ---------- Plot 2: Consistent-feature heatmap ----------

def plot_consistent_heatmap(outcome_col, consistent_features, pred_map,
                            feature_to_category, save_path=None):
    """Heatmap showing only consistent features with labeled rows."""
    valid = [f for f in consistent_features
             if not np.all(np.isnan(pred_map.get(f, np.array([np.nan]))))]
    if not valid:
        return

    trend_map = _compute_trend(pred_map)
    sorted_feats, boundaries = _sort_by_category(
        valid, feature_to_category, CATEGORY_ORDER,
        sort_key=lambda f: trend_map.get(f, 0),
    )

    heat, _ = _build_heatmap_matrix(sorted_feats, pred_map)
    heat_norm = _normalize_heat(heat)
    cmap = make_diverging_cmap()
    n_features = len(sorted_feats)
    fig_height = max(4, n_features * 0.25)

    fig, (ax_cat, ax_heat) = plt.subplots(
        1, 2, figsize=(14, fig_height),
        gridspec_kw={"width_ratios": [0.8, 20], "wspace": 0.02},
    )

    labels = [f"★ {f}" for f in sorted_feats]
    sns.heatmap(heat_norm, cmap=cmap, vmin=-1, vmax=1, center=0,
                yticklabels=labels, xticklabels=False,
                cbar_kws={"label": "Normalized Z-score", "shrink": 0.6},
                ax=ax_heat)
    fs = max(6, min(10, 300 / n_features))
    for lbl in ax_heat.get_yticklabels():
        lbl.set_fontsize(fs)
        lbl.set_color(COLOR_CONSISTENT)

    for start, end, cat in boundaries:
        if start > 0:
            ax_heat.axhline(y=start, color="white", linewidth=2.5)
    ax_heat.set_xlabel(f"{outcome_col} (younger -> older)", fontsize=12)
    ax_heat.set_title(f"{outcome_col}: Consistent Features (★)",
                      fontsize=13, fontweight="bold", pad=12)

    _draw_category_strip(ax_cat, boundaries, n_features, CATEGORY_COLORS)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight", format="pdf")
    plt.close()


# ---------- Plot 3: UpSet intersection plot ----------

def plot_upset(fdr_df, outcome_cols, outcome_labels, save_path=None):
    """UpSet plot showing FDR-significant feature intersections across outcomes."""
    print("\n[UpSet] Plotting")
    n_sets = len(outcome_cols)
    set_names = [outcome_labels.get(oc, oc) for oc in outcome_cols]

    sig_sets = {}
    for oc in outcome_cols:
        col = f"sig_{oc}"
        sig_sets[oc] = set(fdr_df[fdr_df[col]]["feature"]) if col in fdr_df.columns else set()

    combos = []
    for r in range(1, n_sets + 1):
        combos.extend(combinations(range(n_sets), r))

    counts, members_list = [], []
    for combo in combos:
        included = [outcome_cols[i] for i in combo]
        excluded = [outcome_cols[i] for i in range(n_sets) if i not in combo]
        inter = sig_sets[included[0]].copy()
        for oc in included[1:]:
            inter &= sig_sets[oc]
        for oc in excluded:
            inter -= sig_sets[oc]
        counts.append(len(inter))
        members_list.append(combo)

    # Sort by intersection depth (ascending), then by count (descending)
    order = sorted(range(len(combos)),
                   key=lambda i: (len(members_list[i]), -counts[i]))
    pairs = [(counts[i], members_list[i]) for i in order if counts[i] > 0]
    if not pairs:
        print("  No significant intersections")
        return
    sorted_counts, sorted_members = zip(*pairs)

    n_combos = len(sorted_counts)
    fig = plt.figure(figsize=(max(10, n_combos * 0.6 + 4), 7))
    gs = GridSpec(2, 1, height_ratios=[3, 1.5], hspace=0.05, figure=fig)
    ax_bar = fig.add_subplot(gs[0])
    ax_dot = fig.add_subplot(gs[1])

    def _bar_color(m):
        if len(m) == n_sets:  return UPSET_COLORS["all4"]
        if len(m) >= 3:       return UPSET_COLORS["three"]
        if len(m) >= 2:       return UPSET_COLORS["two"]
        return UPSET_COLORS["single"]

    colors = [_bar_color(m) for m in sorted_members]
    x_pos = np.arange(n_combos)
    bars = ax_bar.bar(x_pos, sorted_counts, color=colors, edgecolor="white", linewidth=0.8)
    for i, (cnt, bar) in enumerate(zip(sorted_counts, bars)):
        if cnt > 0:
            ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                        str(cnt), ha="center", va="bottom", fontsize=9)

    ax_bar.set_ylabel("Number of Features", fontsize=11)
    ax_bar.set_xlim(-0.5, n_combos - 0.5)
    ax_bar.set_xticks([])
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["bottom"].set_visible(False)
    ax_bar.set_title("UpSet Plot: FDR-Significant Feature Intersections",
                     fontsize=13, fontweight="bold", pad=12)

    # Dot matrix
    ax_dot.set_xlim(-0.5, n_combos - 0.5)
    ax_dot.set_ylim(-0.5, n_sets - 0.5)
    ax_dot.set_xticks([])
    ax_dot.set_yticks(range(n_sets))
    ax_dot.set_yticklabels(set_names, fontsize=10, fontweight="bold")
    ax_dot.invert_yaxis()
    for sp in ax_dot.spines.values():
        sp.set_visible(False)

    for i, members in enumerate(sorted_members):
        dc = _bar_color(members)
        for j in range(n_sets):
            if j not in members:
                ax_dot.plot(i, j, "o", color="#D3D1C7", markersize=8, zorder=2)
        ml = sorted(members)
        for j in ml:
            ax_dot.plot(i, j, "o", color=dc, markersize=10, zorder=3)
        if len(ml) > 1:
            ax_dot.plot([i, i], [min(ml), max(ml)], color=dc, linewidth=2.5, zorder=1)

    # Legend
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=UPSET_COLORS["all4"],
               markersize=10, label=f"All {n_sets}"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=UPSET_COLORS["three"],
               markersize=10, label="3-way"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=UPSET_COLORS["two"],
               markersize=10, label="2-way"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=UPSET_COLORS["single"],
               markersize=10, label="Single"),
    ]
    ax_bar.legend(handles=legend_elements, loc="upper right", fontsize=8, framealpha=0.9)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight", format="pdf")
    plt.close()


# ---------- Plot 4: 4-column aligned heatmap ----------

def plot_4col_heatmap(outcome_cols, all_pred_maps, feature_to_category,
                      consistent_set, outcome_labels, save_path=None):
    """Four-outcome side-by-side heatmap, sorted by the first outcome's pred[-1]."""
    print("\n[4-col heatmap] Plotting")

    # Collect all valid features across outcomes
    all_features = set()
    for oc in outcome_cols:
        for f, pred in all_pred_maps[oc].items():
            if pred is not None and not np.all(np.isnan(pred)):
                all_features.add(f)
    all_features = list(all_features)

    # Sort by the first outcome's end value
    first_oc = outcome_cols[0]
    first_pm = all_pred_maps[first_oc]

    def end_val(f):
        p = first_pm.get(f)
        if p is not None and not np.all(np.isnan(p)):
            return p[-1]
        return 0.0

    sorted_feats, boundaries = _sort_by_category(
        all_features, feature_to_category, CATEGORY_ORDER, sort_key=end_val,
    )

    n_features = len(sorted_feats)
    n_outcomes = len(outcome_cols)
    cmap = make_diverging_cmap()

    # Build normalized heatmap matrices
    heat_maps_norm = []
    for oc in outcome_cols:
        heat, _ = _build_heatmap_matrix(sorted_feats, all_pred_maps[oc])
        heat_maps_norm.append(_normalize_heat(heat))

    fig_height = max(8, n_features * 0.06)
    width_ratios = [0.6] + [4] * n_outcomes + [6]
    fig, axes = plt.subplots(
        1, n_outcomes + 2, figsize=(32, fig_height),
        gridspec_kw={"width_ratios": width_ratios, "wspace": 0.015},
    )

    ax_cat = axes[0]
    ax_heats = axes[1:1 + n_outcomes]
    ax_mark = axes[-1]

    # Draw heatmap panels
    for col_idx, (oc, hn) in enumerate(zip(outcome_cols, heat_maps_norm)):
        ax = ax_heats[col_idx]
        sns.heatmap(hn, cmap=cmap, vmin=-1, vmax=1, center=0,
                    yticklabels=False, xticklabels=False, cbar=False, ax=ax)
        for s, e, _ in boundaries:
            if s > 0:
                ax.axhline(y=s, color="white", linewidth=2.5)
        ax.set_title(outcome_labels.get(oc, oc), fontsize=12, fontweight="bold", pad=8)

    # Category strip
    _draw_category_strip(ax_cat, boundaries, n_features, CATEGORY_COLORS)

    # Annotate consistent features
    marked = [(i, f) for i, f in enumerate(sorted_feats) if f in consistent_set]
    ax_mark.set_xlim(0, 10)
    ax_mark.set_ylim(n_features, 0)
    ax_mark.set_xticks([])
    ax_mark.set_yticks([])
    for sp in ax_mark.spines.values():
        sp.set_visible(False)

    if marked:
        n_marked = len(marked)
        fs = max(5, min(8, 600 / n_marked))
        gap = max(1.2, n_features / (n_marked * 1.8))
        label_ys = []
        for k, (row_idx, _) in enumerate(marked):
            y = row_idx + 0.5
            if label_ys:
                y = max(y, label_ys[-1] + gap)
            label_ys.append(min(y, n_features - 0.5))
        for k, (row_idx, fname) in enumerate(marked):
            ax_mark.plot([0, 0.8], [row_idx + 0.5, label_ys[k]],
                         color="#8491B4", linewidth=0.5, alpha=0.6)
            ax_mark.text(1.0, label_ys[k], f"★ {fname}", fontsize=fs,
                         color=COLOR_CONSISTENT, fontweight="bold",
                         va="center", ha="left")

    # Colorbar
    sm_cb = ScalarMappable(cmap=cmap, norm=Normalize(vmin=-1, vmax=1))
    sm_cb.set_array([])
    cbar_ax = fig.add_axes([0.08, 0.02, 0.25, 0.01])
    fig.colorbar(sm_cb, cax=cbar_ax, orientation="horizontal").set_label(
        "Normalized Z-score (per-panel scaled)", fontsize=9)

    fig.suptitle(
        "Combined Trend Heatmap: 4 Outcomes (★ = Consistent Across All 4)\n"
        f"Sorted by {outcome_labels.get(first_oc, first_oc)} within each category",
        fontsize=14, fontweight="bold", y=1.02,
    )

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight", format="pdf")
    plt.close()


# ========================= Main =========================

def main():
    t_start = time.time()
    print("=" * 70)
    print("Brain Aging Pipeline (Clean)")
    print("=" * 70)

    # ----- Data loading -----
    df = pd.read_csv(DATA_PATH)
    df = df.loc[:, ~df.columns.duplicated(keep="last")]
    print(f"Data shape: {df.shape}")
    os.makedirs(SAVE_DIR, exist_ok=True)

    feature_to_category = build_feature_category_map(MATCH_PATH, MANUAL_CATEGORY)
    print(f"Feature-to-category mappings: {len(feature_to_category)}")

    exclude = set(OUTCOME_COLS) | COVARIATES_SET | {"avg_outcome"}
    features = select_features(df, feature_to_category, exclude)
    print(f"Valid features: {len(features)}")

    available_covs = [c for c in GLM_COVARIATES if c in df.columns]
    print(f"Covariates: {available_covs}")

    # ----- Step 1: GLM -----
    glm_results = run_glm(df, OUTCOME_COLS, features, available_covs, N_JOBS)

    # ----- Step 2: FDR -----
    consistent_features, fdr_df = run_fdr(glm_results, OUTCOME_COLS, features, FDR_ALPHA)
    consistent_set = set(consistent_features)

    fdr_df.to_csv(os.path.join(SAVE_DIR, "fdr_results.csv"), index=False)
    con_df = fdr_df[fdr_df["consistent"]].copy()
    con_df["Category"] = con_df["feature"].map(feature_to_category)
    con_df.to_csv(os.path.join(SAVE_DIR, "consistent_features.csv"), index=False)

    # ----- Step 3: LOWESS -----
    all_loess, x_grids = run_lowess(df, OUTCOME_COLS, features, GLM_COVARIATES,
                                     n_jobs=N_JOBS)

    # Assemble pred_maps
    all_pred_maps = {}
    for oc in OUTCOME_COLS:
        pred_map = {}
        for f in features:
            if f != oc and f in all_loess[oc]:
                p = all_loess[oc][f]
                pred_map[f] = p.copy() if p is not None else None
        all_pred_maps[oc] = pred_map

    # ----- Step 4: Visualization -----
    print("\n[Visualization]")

    # 4a: Single-outcome heatmaps
    for oc in OUTCOME_COLS:
        feat_list = [f for f in features if f != oc]
        plot_single_heatmap(
            oc, feat_list, all_pred_maps[oc], feature_to_category,
            consistent_set,
            save_path=os.path.join(SAVE_DIR, f"Heatmap_Full_{oc}.pdf"),
        )

    # 4b: Consistent-feature heatmaps
    for oc in OUTCOME_COLS:
        show = [f for f in consistent_features if f in all_pred_maps[oc]]
        if show:
            plot_consistent_heatmap(
                oc, show, all_pred_maps[oc], feature_to_category,
                save_path=os.path.join(SAVE_DIR, f"Heatmap_Consistent_{oc}.pdf"),
            )

    # 4c: UpSet intersection plot
    plot_upset(fdr_df, OUTCOME_COLS, OUTCOME_LABELS,
               save_path=os.path.join(SAVE_DIR, "UpSet_Intersection.pdf"))

    # 4d: 4-column aligned heatmap
    plot_4col_heatmap(
        OUTCOME_COLS, all_pred_maps, feature_to_category,
        consistent_set, OUTCOME_LABELS,
        save_path=os.path.join(SAVE_DIR, "Combined_4col_Heatmap.pdf"),
    )

    print(f"\n{'=' * 70}")
    print(f"Done! Elapsed: {time.time() - t_start:.0f}s")
    print(f"Output: {SAVE_DIR}")
    print(f"{'=' * 70}")


if __name__ == "__main__":
    main()